# Auditoria de qualidade e valida��o - Case Datarisk

## tl;dr

- O gr�o oficial � **uma cobran�a/transa��o por linha**; m�ltiplas transa��es do mesmo cliente e m�s s�o permitidas.
- As chaves dimensionais est�o �ntegras: `ID_CLIENTE` � �nico no cadastro e `(ID_CLIENTE, SAFRA_REF)` � �nico na base mensal. Os joins n�o multiplicam linhas.
- H� **3.931 cobran�as (5,08%)** sem linha correspondente em `base_info`, al�m de nulos parciais nos campos mensais.
- O cadastro cont�m **237 DDDs ausentes, 95 malformados e 8 num�ricos inv�lidos**; eles s�o normalizados e sinalizados pelo pipeline revisado.
- A auditoria remove **71 linhas** com sequ�ncia de datas imposs�vel: 51 prazos inv�lidos e 26 pagamentos anteriores � emiss�o, com 6 casos sobrepostos.
- A valida��o temporal usa 67.622 linhas at� 2021-02 e 9.721 linhas de 2021-03 a 2021-06, sem safras futuras no treino.
- Ap�s as corre��es, o HGB cfg3 segue selecionado: **AUC 0,9403; Gini 0,8807; KS 0,7511; Brier 0,0333; Brier Skill 0,4376**.
- A estabilidade mensal n�o � uniforme: o Brier varia de **0,0282** (mar�o) a **0,0399** (maio). O agregado deve ser apresentado com essa ressalva.


## Contexto e m�todos

Esta auditoria verifica as quatro bases oficiais, a constru��o do alvo, a cardinalidade dos joins, a separa��o temporal e as m�tricas reportadas pelo pipeline. As bases n�o s�o versionadas neste reposit�rio; defina `DATARISK_DATA_DIR` com o diret�rio que cont�m os quatro CSVs.

### Premissas principais

- Inadimpl�ncia: pagamento realizado com **5 dias ou mais** de atraso.
- A base de desenvolvimento termina em 2021-06; mar�o-junho/2021 � o bloco de valida��o fora do tempo.
- Duplicatas exatas nas bases transacionais n�o s�o removidas automaticamente porque o enunciado permite mais de uma transa��o por cliente/m�s e n�o fornece um identificador de transa��o.
- Intervalos de confian�a s�o obtidos por bootstrap em n�vel de cliente para respeitar a depend�ncia entre cobran�as do mesmo cliente.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, roc_auc_score

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "solution.py").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = Path(os.environ.get("DATARISK_DATA_DIR", REPO_ROOT)).resolve()
sys.path.insert(0, str(REPO_ROOT))
import solution

print("C�digo do reposit�rio e bases locais carregados; nenhum CSV � versionado.")

C�digo do reposit�rio e bases locais carregados; nenhum CSV � versionado.


## Dados

In [2]:
cadastral, info, pagamentos_dev, pagamentos_teste = solution.load_data(DATA_DIR)

profile = pd.DataFrame(
    [
        {
            "tabela": name,
            "linhas": len(frame),
            "colunas": len(frame.columns),
            "duplicatas_exatas": int(frame.duplicated().sum()),
            "celulas_nulas": int(frame.isna().sum().sum()),
        }
        for name, frame in {
            "cadastro": cadastral,
            "info": info,
            "pagamentos_dev": pagamentos_dev,
            "pagamentos_teste": pagamentos_teste,
        }.items()
    ]
)
display(profile)

,tabela,linhas,colunas,duplicatas_exatas,celulas_nulas
0,cadastro,1315,8,0,1640
1,info,24401,4,0,1969
2,pagamentos_dev,77414,7,1,1170
3,pagamentos_teste,12275,6,11,131


In [3]:
valid_ddd = set(solution.DDD_REGIAO)
ddd_num = pd.to_numeric(cadastral["DDD"], errors="coerce")
quality = pd.DataFrame(
    {
        "check": [
            "cadastro: ID duplicado",
            "info: cliente-m�s duplicado",
            "DDD ausente",
            "DDD malformado n�o num�rico",
            "DDD num�rico inv�lido",
        ],
        "valor": [
            cadastral.duplicated("ID_CLIENTE").sum(),
            info.duplicated(["ID_CLIENTE", "SAFRA_REF"]).sum(),
            cadastral["DDD"].isna().sum(),
            (cadastral["DDD"].notna() & ddd_num.isna()).sum(),
            (ddd_num.notna() & ~ddd_num.isin(valid_ddd)).sum(),
        ],
    }
)
display(quality)

,check,valor
0,cadastro: ID duplicado,0
1,info: cliente-m�s duplicado,0
2,DDD ausente,237
3,DDD malformado n�o num�rico,95
4,DDD num�rico inv�lido,8


In [4]:
pagamentos_dev = solution.filter_datas_inconsistentes(pagamentos_dev)
dev_basic = solution.add_basic_features(solution.build_target(pagamentos_dev))
train_raw, valid_raw = solution.temporal_split(dev_basic)

joined = (
    dev_basic.assign(_row_id=np.arange(len(dev_basic)))
    .merge(cadastral, on="ID_CLIENTE", how="left", validate="many_to_one", indicator="cadastro_match")
    .merge(info, on=["ID_CLIENTE", "SAFRA_REF"], how="left", validate="many_to_one", indicator="info_match")
)

checks = pd.Series(
    {
        "linhas ap�s filtro de datas": len(dev_basic),
        "treino at�": train_raw["SAFRA_REF"].max(),
        "linhas de treino": len(train_raw),
        "valida��o de": valid_raw["SAFRA_REF"].min(),
        "valida��o at�": valid_raw["SAFRA_REF"].max(),
        "linhas de valida��o": len(valid_raw),
        "linhas ap�s joins": len(joined),
        "cadastro sem correspond�ncia": int((joined["cadastro_match"] == "left_only").sum()),
        "info sem correspond�ncia": int((joined["info_match"] == "left_only").sum()),
    },
    name="resultado",
)
display(checks.to_frame())


Removendo 71 cobranca(s) do desenvolvimento com sequencia de datas inconsistente -- rotulo nao confiavel.


,resultado
linhas ap�s filtro de datas,77343
treino at�,2021-02
linhas de treino,67622
valida��o de,2021-03
valida��o at�,2021-06
linhas de valida��o,9721
linhas ap�s joins,77343
cadastro sem correspond�ncia,0
info sem correspond�ncia,3931


## Resultados

In [5]:
train_df, valid_df = solution.prepare_validation_tables(cadastral, info, pagamentos_dev)
x_train, y_train = train_df[solution.FEATURE_COLUMNS], train_df["INADIMPLENTE"]
x_valid, y_valid = valid_df[solution.FEATURE_COLUMNS], valid_df["INADIMPLENTE"]

models = solution.build_candidate_models(extended=False)
rows = []
for name, model in models.items():
    solution.fit_model(model, x_train, y_train)
    rows.append(solution.evaluate_model(name, model, x_valid, y_valid))

metrics = pd.DataFrame(rows).sort_values(["brier", "auc"], ascending=[True, False])
display(metrics)
best_name = solution.select_best_model(metrics)
best_model = models[best_name]
best_scores = best_model.predict_proba(x_valid)[:, 1]
print(f"Modelo selecionado: {best_name}")

,modelo,auc,gini,ks,average_precision,brier,brier_skill,log_loss,prob_media,target_medio
3,hgb_cfg3,0.940328,0.880655,0.751114,0.656217,0.033282,0.437552,0.121391,0.057548,0.063162
1,hgb_cfg1,0.941369,0.882738,0.755232,0.638765,0.034204,0.421962,0.122745,0.055229,0.063162
2,hgb_cfg2,0.943242,0.886483,0.768171,0.642130,0.034475,0.417380,0.123285,0.052791,0.063162
0,baseline_logistica,0.909307,0.818614,0.710728,0.555110,0.038345,0.351978,0.141501,0.065202,0.063162


Modelo selecionado: hgb_cfg3


In [6]:
monthly = []
for month, group in valid_df.groupby("SAFRA_REF", sort=True):
    row = solution.evaluate_model(str(month), best_model, group[solution.FEATURE_COLUMNS], group["INADIMPLENTE"])
    row.update({"safra": month, "n": len(group)})
    monthly.append(row)

monthly_metrics = pd.DataFrame(monthly)[
    ["safra", "n", "target_medio", "prob_media", "auc", "ks", "average_precision", "brier"]
]
display(monthly_metrics)

,safra,n,target_medio,prob_media,auc,ks,average_precision,brier
0,2021-03,2322,0.065891,0.069161,0.959629,0.800450,0.761300,0.028220
1,2021-04,2358,0.053435,0.052480,0.932072,0.754864,0.592860,0.031396
2,2021-05,2530,0.075494,0.060700,0.936978,0.748427,0.670190,0.039868
3,2021-06,2511,0.057348,0.048392,0.936582,0.765156,0.588353,0.033096


In [7]:
rng = np.random.default_rng(solution.RANDOM_STATE)
bootstrap = valid_df[["ID_CLIENTE", "INADIMPLENTE"]].copy()
bootstrap["score"] = best_scores
groups = {client: group for client, group in bootstrap.groupby("ID_CLIENTE", sort=False)}
clients = np.array(list(groups))
samples = []
for _ in range(300):
    draw = rng.choice(clients, size=len(clients), replace=True)
    sample = pd.concat([groups[client] for client in draw], ignore_index=True)
    samples.append(
        {
            "auc": roc_auc_score(sample["INADIMPLENTE"], sample["score"]),
            "brier": brier_score_loss(sample["INADIMPLENTE"], sample["score"]),
        }
    )

intervals = pd.DataFrame(samples).quantile([0.025, 0.975]).rename(index={0.025: "2,5%", 0.975: "97,5%"})
display(intervals)

,auc,brier
"2,5%",0.917237,0.026021
"97,5%",0.958307,0.040963


## Takeaways

1. **O modelo permanece forte, mas a estimativa pontual n�o deve ser tratada como absoluta.** H� varia��o material entre os meses e incerteza por concentra��o de observa��es nos mesmos clientes.
2. **A baseline log�stica precisa ser n�o balanceada quando Brier � o crit�rio principal.** O balanceamento preservava parte do ranking, mas deslocava a probabilidade m�dia e piorava artificialmente a calibra��o.
3. **A explicabilidade deve usar o mesmo objetivo da sele��o.** A import�ncia por permuta��o passa a usar `neg_brier_score`, em vez da acur�cia padr�o.
4. **Contratos de dados s�o necess�rios antes dos joins.** O pipeline agora bloqueia mudan�as de esquema, duplica��o de chaves dimensionais, datas essenciais ausentes e safras futuras no treino.
5. **Limita��o remanescente:** as configura��es foram escolhidas e reportadas no mesmo bloco temporal de valida��o. Uma pr�xima itera��o deve adotar backtesting temporal em m�ltiplas janelas ou reservar um segundo per�odo rotulado fora da amostra.
